In [1]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
%config InlineBackend.figure_format = 'png'
%config InlineBackend.figure_format = 'retina'

items = pd.read_parquet("items.par")
events = pd.read_parquet("events.par")

# точка разбиения
train_test_global_time_split_date = pd.to_datetime("2017-08-01").date()

train_test_global_time_split_idx = events["started_at"] < train_test_global_time_split_date

events_train = events[train_test_global_time_split_idx]
events_test = events[~train_test_global_time_split_idx]

# количество пользователей в train и test
users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates()
# количество пользователей, которые есть и в train, и в test
common_users = set(users_train).intersection(set(users_test))

print(len(users_train), len(users_test), len(common_users)) 


428220 123223 120858


In [2]:
import scipy
import sklearn.preprocessing

# перекодируем идентификаторы пользователей: 
# из имеющихся в последовательность 0, 1, 2, ...
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events["user_id"])
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

# перекодируем идентификаторы объектов: 
# из имеющихся в последовательность 0, 1, 2, ...
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items["item_id"])
items["item_id_enc"] = item_encoder.transform(items["item_id"])
events_train["item_id_enc"] = item_encoder.transform(events_train["item_id"])
events_test["item_id_enc"] = item_encoder.transform(events_test["item_id"])

/tmp/ipykernel_5437/1044897688.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
/tmp/ipykernel_5437/1044897688.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])
/tmp/ipykernel_5437/1044897688.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See t

In [3]:
# создаём sparse-матрицу формата CSR 
user_item_matrix_train = scipy.sparse.csr_matrix((
    events_train["rating"],
    (events_train['user_id_enc'], events_train['item_id_enc'])),
    dtype=np.int8)

In [4]:
#  код для создания и тренировки модели.
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(factors=50, iterations=50, regularization=0.05, random_state=0)
als_model.fit(user_item_matrix_train)

/home/mle-user/mle_projects/mle-sprint4/mle-recsys-start/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mle-user/mle_projects/mle-sprint4/mle-recsys-start/.venv/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [02:53<00:00,  3.46s/it]


In [45]:
train_item_ids_enc = events_train['item_id_enc'].unique()
max_similar_items = 10

# получаем списки похожих объектов, используя ранее полученную ALS-модель
# метод similar_items возвращает и сам объект, как наиболее похожий
# этот объект мы позже отфильтруем, но сейчас запросим на 1 больше
similar_items = als_model.similar_items(train_item_ids_enc, N=max_similar_items+1)

# преобразуем полученные списки в табличный формат
sim_item_item_ids_enc = similar_items[0]
sim_item_scores = similar_items[1]

similar_items = pd.DataFrame({
    "item_id_enc": train_item_ids_enc,
    "sim_item_id_enc": sim_item_item_ids_enc.tolist(), 
    "score": sim_item_scores.tolist()})

In [46]:
similar_items = similar_items.explode(["sim_item_id_enc", "score"], ignore_index=True)

# приводим типы данных
similar_items["sim_item_id_enc"] = similar_items["sim_item_id_enc"].astype("int")
similar_items["score"] = similar_items["score"].astype("float")

# получаем изначальные идентификаторы
similar_items["item_id_1"] = item_encoder.inverse_transform(similar_items["item_id_enc"])
similar_items["item_id_2"] = item_encoder.inverse_transform(similar_items["sim_item_id_enc"])
similar_items = similar_items.drop(columns=["item_id_enc", "sim_item_id_enc"])

# убираем пары с одинаковыми объектами
similar_items = similar_items.query("item_id_1 != item_id_2")

In [47]:
similar_items.query('item_id_1==7126')

,score,item_id_1,item_id_2
10440,0.948725,7126,7190
10441,0.940997,7126,24280
10442,0.930144,7126,1953
10443,0.925066,7126,58696
10444,0.916340,7126,38296
10445,0.916015,7126,2932
10446,0.913951,7126,7184
10447,0.911433,7126,387749
10448,0.909872,7126,7733
10449,0.909454,7126,30597


In [43]:
items.query('item_id==7126')

,item_id,author,title,description,genre_and_votes,num_pages,average_rating,ratings_count,text_reviews_count,publisher,publication_year,country_code,language_code,format,is_ebook,isbn,isbn13,genre_and_votes_dict,genre_and_votes_str,item_id_enc
738212,7126,"Alexandre Dumas, Robin Buss",The Count of Monte Cristo,'On what slender threads do life and fortune h...,"{'Classics': 18271, 'Fiction': 6761, 'Historic...",1276,4.22,564778,12165,Penguin Classics,2003,US,eng,Paperback,False,0140449264,9780140449266,"{'Academic': None, 'Academic-Academia': None, ...","Classics 18271, Fiction 6761, Historical-Histo...",839


In [48]:
similar_items.to_parquet("similar_items.parquet") 

In [49]:
# можно оценить глазами списки похожих объектов для каких-то уже известных
def print_sim_items(item_id, similar_items):

    item_columns_to_use = ["item_id", "author", "title", "genre_and_votes", "average_rating", "ratings_count"]
    
    item_id_1 = items.query("item_id == @item_id")[item_columns_to_use]
    display(item_id_1)
    
    si = similar_items.query("item_id_1 == @item_id")
    si = si.merge(items[item_columns_to_use].set_index("item_id"), left_on="item_id_2", right_index=True)
    display(si)

In [52]:
print_sim_items(3, similar_items) 

,item_id,author,title,genre_and_votes,average_rating,ratings_count
1584855,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,"{'Fantasy': 59818, 'Fiction': 17918, 'Young Ad...",4.45,4765497


,score,item_id_1,item_id_2,author,title,genre_and_votes,average_rating,ratings_count
188,0.986763,3,15881,"J.K. Rowling, Mary GrandPré",Harry Potter and the Chamber of Secrets (Harry...,"{'Fantasy': 50130, 'Young Adult': 15202, 'Fict...",4.38,1821802
189,0.974947,3,5,"J.K. Rowling, Mary GrandPré",Harry Potter and the Prisoner of Azkaban (Harr...,"{'Fantasy': 49784, 'Young Adult': 15393, 'Fict...",4.53,1876252
190,0.954390,3,6,"J.K. Rowling, Mary GrandPré",Harry Potter and the Goblet of Fire (Harry Pot...,"{'Fantasy': 48257, 'Young Adult': 15483, 'Fict...",4.53,1792561
191,0.934225,3,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Order of the Phoenix (Har...,"{'Fantasy': 46485, 'Young Adult': 15194, 'Fict...",4.47,1766895
192,0.922894,3,1,J.K. Rowling,Harry Potter and the Half-Blood Prince (Harry ...,"{'Fantasy': 46400, 'Young Adult': 15083, 'Fict...",4.54,1713866
193,0.907875,3,136251,J.K. Rowling,Harry Potter and the Deathly Hallows (Harry Po...,"{'Fantasy': 46667, 'Young Adult': 15403, 'Fict...",4.62,1784684
194,0.861305,3,8388506,"Bruno Nogueira, João Quadros","Tubo de Ensaio, Parte II","{'Humor': 4, 'Humor-Comedy': 1}",3.26,39
195,0.861305,3,6379485,"Bruno Nogueira, João Quadros",Tubo de Ensaio,"{'Humor': 5, 'Humor-Comedy': 2}",3.27,44
196,0.838405,3,7904207,Jim Henry,Antiquity Calais: Standing at Armageddon (The ...,None,4.61,16
197,0.737723,3,8226034,Hans Scherfig,Frydenholm,"{'Historical-Historical Fiction': 3, 'Fiction'...",4.06,98


In [53]:
similar_items

,score,item_id_1,item_id_2
1,0.986732,11012,7588
2,0.964650,11012,4900
3,0.963963,11012,19117
4,0.962113,11012,10975
5,0.962032,11012,87346
...,...,...,...
456209,0.925937,748350,330721
456210,0.923935,748350,228926
456211,0.921570,748350,709835
456212,0.921420,748350,114173


In [61]:
events[events['user_id']==1000000][:10]['item_id']

6679625       11012
6679617        4671
6679618           5
6679620           2
6679621       14497
6679622       28921
6679623    15745753
6679616       15881
6679624        4692
6679627        6334
Name: item_id, dtype: int64

In [56]:
user_events

NameError: name 'user_events' is not defined